<a href="https://colab.research.google.com/github/Rishy-09/CNLP_LAB_SEM7/blob/main/Notebooks/Experiment5/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Experiment 5: Subword Tokenization and POS Tagging

### Objective
To implement advanced subword-level tokenization utilizing BPE and SentencePiece architectures, and to perform Part-of-Speech (POS) tagging for linguistic syntax analysis using NLP toolkits.

### Procedure & Implementation
This experiment focuses on breaking down texts into subword units and categorizing tokens by their grammatical roles. We will explore both pretrained models and training from scratch for tokenization, and compare different NLP toolkits for POS tagging.

### 0. Setup: Install Libraries and Download Models

Before we begin with the experiment, we need to ensure all necessary Python libraries (like `transformers`, `tokenizers`, `sentencepiece`, `spacy`, `nltk`) are installed, and any required language models (like spaCy's `en_core_web_sm`) are downloaded. The `%%capture` magic command is used to suppress the installation output, keeping the notebook clean.

In [1]:
%%capture
!pip install transformers tokenizers sentencepiece spacy nltk
!python -m spacy download en_core_web_sm

### 1. Create Input Data File

We'll create a sample text file `input_sub_word_data.txt` which will be used for training tokenizers from scratch and for POS tagging frequency analysis.

### Wisdom:
Creating a dedicated input file for text data ensures that all subsequent operations (especially tokenizer training) use a consistent source. For larger texts, this approach is more memory-efficient than loading the entire text into a Python string at once.

In [2]:
sample_data = [
    "The quick brown fox jumps over the lazy dog. This is a very interesting sentence for NLP.",
    "Natural Language Processing is a field of artificial intelligence that focuses on the interaction between computers and human language.",
    "Subword tokenization helps in handling out-of-vocabulary words and reduces vocabulary size.",
    "SentencePiece is a language-agnostic subword tokenizer.",
    "Part-of-Speech tagging is crucial for understanding the grammatical structure of sentences.",
    "Different NLP libraries might use different tag sets for POS tagging.",
    "The cat sat on the mat. The dog chased the cat. The bird flew away."
]

with open('input_sub_word_data.txt', 'w', encoding='utf-8') as f:
    for line in sample_data:
        f.write(line + '\n')

print("input_sub_word_data.txt created successfully.")

with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    sample_text = f.read().strip().split('\n')[0] # Using first line as sample

print(f"\nSample text for tokenization: '{sample_text}'")

input_sub_word_data.txt created successfully.

Sample text for tokenization: 'The quick brown fox jumps over the lazy dog. This is a very interesting sentence for NLP.'


### 5.1 BPE Tokenization

#### a. Using a Pretrained Model (GPT-2 uses BPE)

### Procedure: Pretrained BPE Tokenization
Loading a GPT-2 tokenizer (`GPT2Tokenizer.from_pretrained()`) to encode and convert sample text into subword identifiers and tokens (`encode()`, `convert_ids_to_tokens()`).

### Observations & Learnings:
Pretrained BPE tokenizers like GPT-2's are designed to handle a wide range of text due to their training on massive datasets. They are effective at resolving out-of-vocabulary (OOV) issues by breaking down rare words into frequent character sub-sequences. The output will show how common words or parts of words are represented by unique IDs, often with a special prefix (like `Ġ` or ` `) to indicate word boundaries.

In [3]:
from transformers import GPT2Tokenizer

tokenizer_bpe_pre = GPT2Tokenizer.from_pretrained('gpt2')

print('--- BPE Pretrained ---')
encoded_bpe = tokenizer_bpe_pre.encode(sample_text)
tokens_bpe = tokenizer_bpe_pre.convert_ids_to_tokens(encoded_bpe)
print(f'Encoded IDs: {encoded_bpe[:20]}')
print(f'Tokens: {tokens_bpe[:20]}')
for t, i in zip(tokens_bpe[:20], encoded_bpe[:20]):
    print(f'{t}: {i}')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

--- BPE Pretrained ---
Encoded IDs: [464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13, 770, 318, 257, 845, 3499, 6827, 329, 399, 19930, 13]
Tokens: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.', 'ĠThis', 'Ġis', 'Ġa', 'Ġvery', 'Ġinteresting', 'Ġsentence', 'Ġfor', 'ĠN', 'LP', '.']
The: 464
Ġquick: 2068
Ġbrown: 7586
Ġfox: 21831
Ġjumps: 18045
Ġover: 625
Ġthe: 262
Ġlazy: 16931
Ġdog: 3290
.: 13
ĠThis: 770
Ġis: 318
Ġa: 257
Ġvery: 845
Ġinteresting: 3499
Ġsentence: 6827
Ġfor: 329
ĠN: 399
LP: 19930
.: 13


#### b. Without a Pretrained Model (Train from scratch)

### Procedure: BPE Tokenization from Scratch
Instantiating a native tokenizer (`Tokenizer(BPE())`), applying whitespace pre-tokenization (`Whitespace()`), and training it on the provided text file (`BpeTrainer()`, `train()`).

### Observations & Learnings:
Training a BPE tokenizer from scratch allows for custom vocabulary induction, specifically tailored to the nuances of the training corpus. This can be beneficial for domain-specific tasks where a generic pretrained tokenizer might not be optimal. The vocabulary size (`vocab_size`) is a crucial hyperparameter that balances token granularity and vocabulary size.

In [4]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer_bpe_scratch = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer_bpe_scratch.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=1000, special_tokens=['[UNK]'])
tokenizer_bpe_scratch.train(['input_sub_word_data.txt'], trainer)

print('\n--- BPE From Scratch ---')
encoded_scratch = tokenizer_bpe_scratch.encode(sample_text)
print(f'Encoded IDs: {encoded_scratch.ids[:20]}')
print(f'Tokens: {encoded_scratch.tokens[:20]}')
for t, i in zip(encoded_scratch.tokens[:20], encoded_scratch.ids[:20]):
    print(f'{t}: {i}')


--- BPE From Scratch ---
Encoded IDs: [50, 238, 245, 226, 232, 167, 48, 234, 108, 2, 218, 47, 10, 207, 249, 95, 59, 74, 2]
Tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.', 'This', 'is', 'a', 'very', 'interesting', 'sentence', 'for', 'NLP', '.']
The: 50
quick: 238
brown: 245
fox: 226
jumps: 232
over: 167
the: 48
lazy: 234
dog: 108
.: 2
This: 218
is: 47
a: 10
very: 207
interesting: 249
sentence: 95
for: 59
NLP: 74
.: 2


### 5.2 SentencePiece Tokenization

#### a. Using a Pretrained Model (T5 uses SentencePiece)

### Procedure: Pretrained SentencePiece Tokenization
Utilizing a T5 tokenizer (`T5Tokenizer.from_pretrained()`) to segment the text into SentencePiece subword elements (`encode()`, `convert_ids_to_tokens()`).

### Observations & Learnings:
SentencePiece is known for its language-agnostic string tokenization, which is particularly useful for languages without clear word boundaries. Pretrained models like T5's leverage this to provide robust subword units. You'll often see a special underscore prefix (` ` or `▁`) indicating the start of a new word, as SentencePiece treats the entire input as a raw string.

In [5]:
from transformers import T5Tokenizer

tokenizer_sp_pre = T5Tokenizer.from_pretrained('t5-small', legacy=False)

print('\n--- SentencePiece Pretrained ---')
encoded_sp = tokenizer_sp_pre.encode(sample_text)
tokens_sp = tokenizer_sp_pre.convert_ids_to_tokens(encoded_sp)
print(f'Encoded IDs: {encoded_sp[:20]}')
print(f'Tokens: {tokens_sp[:20]}')
for t, i in zip(tokens_sp[:20], encoded_sp[:20]):
    print(f'{t}: {i}')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]


--- SentencePiece Pretrained ---
Encoded IDs: [37, 1704, 4216, 3, 20400, 4418, 7, 147, 8, 19743, 1782, 5, 100, 19, 3, 9, 182, 1477, 7142, 21]
Tokens: ['▁The', '▁quick', '▁brown', '▁', 'fox', '▁jump', 's', '▁over', '▁the', '▁lazy', '▁dog', '.', '▁This', '▁is', '▁', 'a', '▁very', '▁interesting', '▁sentence', '▁for']
▁The: 37
▁quick: 1704
▁brown: 4216
▁: 3
fox: 20400
▁jump: 4418
s: 7
▁over: 147
▁the: 8
▁lazy: 19743
▁dog: 1782
.: 5
▁This: 100
▁is: 19
▁: 3
a: 9
▁very: 182
▁interesting: 1477
▁sentence: 7142
▁for: 21


#### b. Without a Pretrained Model (Train from scratch)

### Procedure: SentencePiece Tokenization from Scratch
Training a native SentencePiece model directly on the raw text file (`spm.SentencePieceTrainer.train()`) and encoding the string via the trained processor (`spm.SentencePieceProcessor()`, `encode_as_pieces()`).

### Observations & Learnings:
Similar to BPE from scratch, training SentencePiece from scratch provides a custom tokenizer. The `vocab_size` here also dictates the balance between token detail and vocabulary size. This method allows for fine-tuning the tokenization process for specific dataset characteristics, which can be critical for model performance in downstream tasks.

In [8]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(input='input_sub_word_data.txt', model_prefix='m_sp', vocab_size=128)
sp = spm.SentencePieceProcessor(model_file='m_sp.model')

print('\n--- SentencePiece From Scratch ---')
encoded_sp_scratch = sp.encode_as_ids(sample_text)
tokens_sp_scratch = sp.encode_as_pieces(sample_text)
print(f'Encoded IDs: {encoded_sp_scratch[:20]}')
print(f'Tokens: {tokens_sp_scratch[:20]}')
for t, i in zip(tokens_sp_scratch[:20], encoded_sp_scratch[:20]):
    print(f'{t}: {i}')


--- SentencePiece From Scratch ---
Encoded IDs: [12, 3, 100, 27, 28, 99, 21, 58, 11, 26, 48, 101, 3, 98, 52, 54, 64, 37, 7, 109]
Tokens: ['▁The', '▁', 'q', 'u', 'ic', 'k', '▁b', 'ro', 'w', 'n', '▁fo', 'x', '▁', 'j', 'um', 'ps', '▁o', 'ver', '▁the', '▁la']
▁The: 12
▁: 3
q: 100
u: 27
ic: 28
k: 99
▁b: 21
ro: 58
w: 11
n: 26
▁fo: 48
x: 101
▁: 3
j: 98
um: 52
ps: 54
▁o: 64
ver: 37
▁the: 7
▁la: 109


### 5.3 POS Tagging (Short Sentence)

### Wisdom:
Part-of-Speech (POS) tagging is a fundamental NLP task that assigns a grammatical category (e.g., noun, verb, adjective) to each word in a text. It is a crucial prerequisite for many advanced NLP tasks such as syntactic parsing, named entity recognition, and machine translation. Analyzing POS tags helps us understand the grammatical structure and meaning of sentences.

In [9]:
sentence = 'The young student is reading an interesting book in the library.'

print(f"\nSentence for POS Tagging: '{sentence}'")


Sentence for POS Tagging: 'The young student is reading an interesting book in the library.'


#### a. Using spaCy

### Procedure: POS Tagging using spaCy
Processing the sentence through the `en_core_web_sm` pipeline (`nlp(sentence)`) and iterating over tokens to extract and explain their POS tags (`token.pos_`, `spacy.explain()`).

### Observations & Learnings:
spaCy provides highly efficient and accurate POS tagging with detailed explanations for each tag. Its pipeline approach makes it easy to access linguistic annotations. The consistency and rich metadata provided by spaCy are invaluable for deep linguistic analysis.

In [10]:
import spacy

nlp = spacy.load('en_core_web_sm')
doc = nlp(sentence)

print('\n--- POS Tagging (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-'*50)
for token in doc:
    print(f'{token.text:<15} | {token.pos_:<10} | {spacy.explain(token.pos_)}')


--- POS Tagging (spaCy) ---
Token           | POS Tag    | Description
--------------------------------------------------
The             | DET        | determiner
young           | ADJ        | adjective
student         | NOUN       | noun
is              | AUX        | auxiliary
reading         | VERB       | verb
an              | DET        | determiner
interesting     | ADJ        | adjective
book            | NOUN       | noun
in              | ADP        | adposition
the             | DET        | determiner
library         | NOUN       | noun
.               | PUNCT      | punctuation


#### b. Using NLTK

### Procedure: POS Tagging using NLTK
Utilizing NLTK's standard word tokenizer (`nltk.word_tokenize()`) followed by the perceptron tagger (`nltk.pos_tag()`) to identify part-of-speech roles.

### Observations & Learnings:
NLTK is a classic NLP library, and its POS tagger is widely used. One key difference compared to spaCy is that NLTK often uses a different, more granular tag set (e.g., Penn Treebank tags). It's important to be aware of these toolkit discrepancies, as different libraries might employ divergent tag sets and underlying models, resulting in slightly varied POS classifications for identical text.

In [13]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True) # Added this line

tokens_nltk = nltk.word_tokenize(sentence)
pos_tags_nltk = nltk.pos_tag(tokens_nltk)

print('\n--- POS Tagging (NLTK) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Description"}')
print('-'*50)
for token, tag in pos_tags_nltk:
    # NLTK tags are sometimes different, using the tag itself as description for simplicity here
    print(f'{token:<15} | {tag:<10} | {tag}')


--- POS Tagging (NLTK) ---
Token           | POS Tag    | Description
--------------------------------------------------
The             | DT         | DT
young           | JJ         | JJ
student         | NN         | NN
is              | VBZ        | VBZ
reading         | VBG        | VBG
an              | DT         | DT
interesting     | JJ         | JJ
book            | NN         | NN
in              | IN         | IN
the             | DT         | DT
library         | NN         | NN
.               | .          | .


### 5.4 POS Tagging with Frequency (Large File)

### Procedure: POS Tagging with Frequency Analysis
Parsing a large unstructured text file via the spaCy pipeline (`nlp(text)`). Aggregating POS tag frequencies alongside the tokens themselves utilizing standard collection modules (`Counter()`). Extracting the most prominent syntactical elements (`most_common(20)`).

### Observations & Learnings:
Analyzing POS tag frequencies across a larger corpus helps in understanding the overall syntactical structure and common linguistic patterns within the text. This can reveal insights into the writing style, domain-specific terminology, and the distribution of different grammatical classes. The `Counter` object is an efficient way to tally occurrences and quickly identify the most frequent elements, providing a macro-level view of the text's composition.

In [14]:
from collections import Counter

with open('input_sub_word_data.txt', 'r', encoding='utf-8') as f:
    large_text = f.read()

# Process first 5000 chars for demonstration (or entire text if shorter)
doc_large = nlp(large_text[:min(len(large_text), 5000)])

# Count frequencies of (Token, POS, Description)
pos_freq = Counter([(token.text, token.pos_, spacy.explain(token.pos_)) for token in doc_large if not token.is_space])

print('\n--- POS Tagging with Frequency (spaCy) ---')
print(f'{"Token":<15} | {"POS Tag":<10} | {"Frequency":<10} | {"Description"}')
print('-'*65)
for (token, pos, desc), freq in pos_freq.most_common(20):
    print(f'{token:<15} | {pos:<10} | {freq:<10} | {desc}')


--- POS Tagging with Frequency (spaCy) ---
Token           | POS Tag    | Frequency  | Description
-----------------------------------------------------------------
.               | PUNCT      | 10         | punctuation
the             | DET        | 5          | determiner
-               | PUNCT      | 5          | punctuation
The             | DET        | 4          | determiner
is              | AUX        | 4          | auxiliary
of              | ADP        | 4          | adposition
a               | DET        | 3          | determiner
for             | ADP        | 3          | adposition
dog             | NOUN       | 2          | noun
NLP             | PROPN      | 2          | proper noun
on              | ADP        | 2          | adposition
and             | CCONJ      | 2          | coordinating conjunction
language        | NOUN       | 2          | noun
tagging         | NOUN       | 2          | noun
cat             | NOUN       | 2          | noun
quick           |